In [3]:
import numpy as np
import pandas as pd
from lifelines import KaplanMeierFitter
from lifelines.utils import restricted_mean_survival_time

## Read data

In [4]:
data_ct = pd.read_csv("outputs/trophos_ct_loo_covariates_diff_datasets.csv")

## Compute ATE and variance in the classical case

In [5]:
tau = 1.5

kmf_trt = KaplanMeierFitter().fit(data_ct[data_ct['TREATMENT']==1]["TIME_OF_EVENT"],data_ct[data_ct['TREATMENT']==1]["EVENT"])
rmst_trt, var_rmst_trt = restricted_mean_survival_time(kmf_trt, t=tau, return_variance=True)

kmf_con = KaplanMeierFitter().fit(data_ct[data_ct['TREATMENT']==0]["TIME_OF_EVENT"],data_ct[data_ct['TREATMENT']==0]["EVENT"])
rmst_con, var_rmst_con = restricted_mean_survival_time(kmf_con, t=tau, return_variance=True)

/Users/maylis.tran/miniforge3/envs/leaspy_v2/lib/python3.11/site-packages/lifelines/utils/__init__.py:320: IntegrationWarning: The maximum number of subdivisions (50) has been achieved.
  If increasing the limit yields no improvement it is advised to analyze 
  the integrand in order to determine the difficulties.  If the position of a 
  local difficulty can be determined (singularity, discontinuity) one will 
  probably gain from splitting up the interval and calling the integrator 
  on the subranges.  Perhaps a special-purpose integrator should be used.
  return 2 * quad(lambda tau: (tau * model.predict(tau)), 0, t, epsabs=1.49e-10, epsrel=1e-10)[0]
/Users/maylis.tran/miniforge3/envs/leaspy_v2/lib/python3.11/site-packages/lifelines/utils/__init__.py:320: IntegrationWarning: The maximum number of subdivisions (50) has been achieved.
  If increasing the limit yields no improvement it is advised to analyze 
  the integrand in order to determine the difficulties.  If the position of a 

In [6]:
# Compute classic ATE using RMST 
m = len(data_ct[data_ct['TREATMENT'] == 1])
n = len(data_ct[data_ct['TREATMENT'] == 0])

ate_classic_1 = rmst_trt - rmst_con
var_classic_1 = var_rmst_trt/m + var_rmst_con/n

var_rmst_trt_loo = data_ct[data_ct['TREATMENT'] == 1]['LOO_RMST_SEP'].var()
var_rmst_con_loo = data_ct[data_ct['TREATMENT'] == 0]['LOO_RMST_SEP'].var()

ate_classic_2 = data_ct[data_ct['TREATMENT'] == 1]['LOO_RMST_SEP'].mean() - data_ct[data_ct['TREATMENT'] == 0]['LOO_RMST_SEP'].mean()
var_classic_2 = var_rmst_trt_loo/m + var_rmst_con_loo/n

print("Average Treatment Effect (ATE) with RMST:", ate_classic_1, ", ATE with LOO RMST:", ate_classic_2)
print("Variance of ATE with RMST:", var_classic_1, ", with LOO RMST", var_classic_2)
print("Confidence interval of ATE with RMST:", ate_classic_2 - 1.96 * np.sqrt(var_classic_2), "to", ate_classic_2 + 1.96 * np.sqrt(var_classic_2))
print("Confidence interval width with RMST:", - (ate_classic_2 - 1.96 * np.sqrt(var_classic_2)) + ate_classic_2 + 1.96 * np.sqrt(var_classic_2))

Average Treatment Effect (ATE) with RMST: 0.016257131093417865 , ATE with LOO RMST: 0.016257131093678545
Variance of ATE with RMST: 0.0008282106114065249 , with LOO RMST 0.0008803823558920516
Confidence interval of ATE with RMST: -0.04189849510357691 to 0.074412757290934
Confidence interval width with RMST: 0.11631125239451091


## Apply PPCT linear 

In [7]:
col = 'LOO_RMST_PRED_COX_PROACT_ANSWERALS'
sigma_f_2 = data_ct[col].var()
sigma_t_2 = var_rmst_trt_loo
sigma_c_2 = var_rmst_con_loo

rho_t = np.cov(
    data_ct.loc[data_ct['TREATMENT'] == 1, col],
    data_ct.loc[data_ct['TREATMENT'] == 1, 'LOO_RMST_SEP']
)[0, 1] / np.sqrt(sigma_f_2 * sigma_t_2)

rho_c = np.cov(
    data_ct.loc[data_ct['TREATMENT'] == 0, col],
    data_ct.loc[data_ct['TREATMENT'] == 0, 'LOO_RMST_SEP']
)[0, 1] / np.sqrt(sigma_f_2 * sigma_c_2)
lambda_star = 0.9393584736465697

# PPI Variance
var_ppi = ((1/m) * (sigma_t_2 + (lambda_star**2)*sigma_f_2 - 2*lambda_star*np.sqrt(sigma_f_2*sigma_t_2)*rho_t)
            + (1/n) * (sigma_c_2 + (lambda_star**2)*sigma_f_2 - 2*lambda_star*np.sqrt(sigma_f_2*sigma_c_2)*rho_c))

# PPI ATE
ate_ppi = (
    (data_ct.loc[data_ct['TREATMENT'] == 1, 'LOO_RMST_SEP'] - lambda_star * data_ct.loc[data_ct['TREATMENT'] == 1, col]).mean()
    - (data_ct.loc[data_ct['TREATMENT'] == 0, 'LOO_RMST_SEP'] - lambda_star * data_ct.loc[data_ct['TREATMENT'] == 0, col]).mean()
)

# R²
r_2 = data_ct['LOO_RMST_SEP'].corr(data_ct[col]) ** 2
var_r2 = var_classic_1 * (1 - r_2)
print(ate_ppi, var_ppi)

0.02019558945256472 0.0008595285801250247


In [8]:
results = []

for col in [c for c in data_ct.columns if c.startswith("LOO_RMST_PRED_")]:
    model_name = col.replace("LOO_RMST_PRED_", "") 
    
    sigma_f_2 = data_ct[col].var()
    sigma_t_2 = var_rmst_trt_loo
    sigma_c_2 = var_rmst_con_loo

    rho_t = np.cov(
        data_ct.loc[data_ct['TREATMENT'] == 1, col],
        data_ct.loc[data_ct['TREATMENT'] == 1, 'LOO_RMST_SEP']
    )[0, 1] / np.sqrt(sigma_f_2 * sigma_t_2)
    
    rho_c = np.cov(
        data_ct.loc[data_ct['TREATMENT'] == 0, col],
        data_ct.loc[data_ct['TREATMENT'] == 0, 'LOO_RMST_SEP']
    )[0, 1] / np.sqrt(sigma_f_2 * sigma_c_2)

    lambda_star = (n * np.sqrt(sigma_t_2) * rho_t + m * np.sqrt(sigma_c_2) * rho_c) / ((n + m) * np.sqrt(sigma_f_2))
    #lambda_star = 0.083


    # PPI Variance
    var_ppi = ((1/m) * (sigma_t_2 + (lambda_star**2)*sigma_f_2 - 2*lambda_star*np.sqrt(sigma_f_2*sigma_t_2)*rho_t)
              + (1/n) * (sigma_c_2 + (lambda_star**2)*sigma_f_2 - 2*lambda_star*np.sqrt(sigma_f_2*sigma_c_2)*rho_c))

    # PPI ATE
    ate_ppi = (
        (data_ct.loc[data_ct['TREATMENT'] == 1, 'LOO_RMST_SEP'] - lambda_star * data_ct.loc[data_ct['TREATMENT'] == 1, col]).mean()
        - (data_ct.loc[data_ct['TREATMENT'] == 0, 'LOO_RMST_SEP'] - lambda_star * data_ct.loc[data_ct['TREATMENT'] == 0, col]).mean()
    )

    # R²
    r_2 = data_ct['LOO_RMST_SEP'].corr(data_ct[col]) ** 2
    var_r2 = var_classic_1 * (1 - r_2)

    # Append results
    results.append({
        "Model": model_name,
        "PPI_ATE": ate_ppi,
        "PPI_Variance": var_ppi,
        "Lambda_star": lambda_star,
        "R2": r_2,
        "PPI_Var_R2_formula": var_r2,
        "Number of patients possibl to remove": int(r_2*(m + n))
    })

df_ppi_summary = pd.DataFrame(results).set_index("Model")
df_ppi_summary = df_ppi_summary.sort_values(by="PPI_Variance", ascending=True)
df_ppi_summary


,PPI_ATE,PPI_Variance,Lambda_star,R2,PPI_Var_R2_formula,Number of patients possibl to remove
Model,,,,,,
POLY_PROACT_PULSE_ANSWERALS,0.017737,0.000792,0.898866,0.097971,0.000747,49
POLY_PROACT_ANSWERALS,0.016973,0.000795,0.888974,0.095104,0.000749,48
POLY_PROACT_PULSE,0.017675,0.000799,0.734461,0.090575,0.000753,46
POLY_PULSE_ANSWERALS,0.017480,0.000800,0.902070,0.089422,0.000754,45
POLY_PROACT_PULSE_NEUROBANK_ANSWERALS,0.014432,0.000807,0.789877,0.081587,0.000761,41
POLY_PROACT_PULSE_NEUROBANK,0.013140,0.000809,0.750613,0.079414,0.000762,40
POLY_ANSWERALS,0.015955,0.000811,0.853594,0.077117,0.000764,39
POLY_PROACT_NEUROBANK_ANSWERALS,0.014326,0.000811,0.762790,0.077018,0.000764,39
POLY_PULSE_NEUROBANK_ANSWERALS,0.013384,0.000814,0.748286,0.073746,0.000767,37


In [9]:
print(df_ppi_summary.to_latex(escape=True))

\begin{tabular}{lrrrrrr}
\toprule
 & PPI\_ATE & PPI\_Variance & Lambda\_star & R2 & PPI\_Var\_R2\_formula & Number of patients possibl to remove \\
Model &  &  &  &  &  &  \\
\midrule
POLY\_PROACT\_PULSE\_ANSWERALS & 0.017737 & 0.000792 & 0.898866 & 0.097971 & 0.000747 & 49 \\
POLY\_PROACT\_ANSWERALS & 0.016973 & 0.000795 & 0.888974 & 0.095104 & 0.000749 & 48 \\
POLY\_PROACT\_PULSE & 0.017675 & 0.000799 & 0.734461 & 0.090575 & 0.000753 & 46 \\
POLY\_PULSE\_ANSWERALS & 0.017480 & 0.000800 & 0.902070 & 0.089422 & 0.000754 & 45 \\
POLY\_PROACT\_PULSE\_NEUROBANK\_ANSWERALS & 0.014432 & 0.000807 & 0.789877 & 0.081587 & 0.000761 & 41 \\
POLY\_PROACT\_PULSE\_NEUROBANK & 0.013140 & 0.000809 & 0.750613 & 0.079414 & 0.000762 & 40 \\
POLY\_ANSWERALS & 0.015955 & 0.000811 & 0.853594 & 0.077117 & 0.000764 & 39 \\
POLY\_PROACT\_NEUROBANK\_ANSWERALS & 0.014326 & 0.000811 & 0.762790 & 0.077018 & 0.000764 & 39 \\
POLY\_PULSE\_NEUROBANK\_ANSWERALS & 0.013384 & 0.000814 & 0.748286 & 0.073746 & 0.000767 &